# Explainable AI for Heavy Rainfall Event Prediction

This notebook is a clean companion to the scripted project. It follows the same workflow used by the repository scripts: generate a synthetic meteorological dataset, train a Random Forest Classifier, evaluate the model, and interpret predictions with SHAP.

## Important Dataset Note

The dataset used here is synthetic. It is designed for reproducible research-code demonstration and should not be described as real operational rainfall data.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT / 'src'))

from generate_dataset import build_synthetic_dataset, DATA_PATH

data = build_synthetic_dataset()
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
data.to_csv(DATA_PATH, index=False)
data.head()

In [ ]:
data['heavy_rainfall_event'].value_counts().sort_index()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, roc_auc_score

FEATURE_COLUMNS = ['temperature', 'humidity', 'cloud_cover', 'historical_rainfall']
TARGET_COLUMN = 'heavy_rainfall_event'

X = data[FEATURE_COLUMNS]
y = data[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

rf = RandomForestClassifier(
    n_estimators=500,
    class_weight='balanced',
    random_state=42,
)
rf.fit(X_train, y_train)

predictions = rf.predict(X_test)
probabilities = rf.predict_proba(X_test)[:, 1]

print(classification_report(y_test, predictions, target_names=['No heavy rainfall', 'Heavy rainfall']))
print('Accuracy:', accuracy_score(y_test, predictions))
print('ROC-AUC:', roc_auc_score(y_test, probabilities))
print('Confusion matrix:')
print(confusion_matrix(y_test, predictions))

In [ ]:
import shap
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    class_1_shap_values = shap_values[1]
elif getattr(shap_values, 'ndim', None) == 3:
    class_1_shap_values = shap_values[:, :, 1]
else:
    class_1_shap_values = shap_values

shap.summary_plot(class_1_shap_values, X_test, show=False)
plt.title('SHAP Summary: Heavy Rainfall Event Prediction')
plt.tight_layout()
plt.show()